# 🧬 BLAST Runner
Interactive BLAST search interface — supports pre-built BLAST databases **and** raw FASTA files (auto-indexed on first run).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  DATABASE REGISTRY  —  edit this cell to add your databases     ║
# ║                                                                  ║
# ║  Each entry:  "Display name": "path"                            ║
# ║  Path can be:                                                    ║
# ║    • A pre-built BLAST db prefix  →  /data/nr  (no extension)   ║
# ║    • A FASTA file                 →  /data/seqs.fasta            ║
# ║      (makeblastdb is run automatically on first use)             ║
# ╚══════════════════════════════════════════════════════════════════╝f

import os, tempfile, subprocess
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
import numpy as np
import io, base64
from datetime import datetime
from pathlib import Path

SCRIPT_DIR = Path.cwd()
BINDIR = SCRIPT_DIR.parent
BASEDIR = BINDIR.parent

DBDIR = BASEDIR / "database"

collated = (
    DBDIR
    / "collated"
    / "Version1"
    / "filtered"
    / "85comp10con"
)

collated_ffn = (
    collated
    / "fna"
    / "v1_cp85_con10_dna.ffn"
)

collated_protein = (
    collated
    / "fasta"
    / "v1_cp85_con10.fasta"
)

GENOME = ""
GENOME_PATH = (
    DBDIR
    / "cds_genomes"
    / f"{GENOME}_genomic"
    / f"{GENOME}_genomic.faa"
)

DATABASES = {
    # ── Pre-built BLAST databases ──────────────────────────────────
    "nr  (NCBI non-redundant protein)": "/path/to/blast/db/nr",
    "nt  (NCBI nucleotide)": "/path/to/blast/db/nt",
    "swissprot": "/path/to/blast/db/swissprot",

    # ── Raw FASTA files (auto-indexed) ─────────────────────────────
    "Asgard Proteins (FAA)": collated_protein,
    "Asgard CDS (FFN) ": collated_ffn,
    "My genome (FASTA)": "/path/to/genome.fna",

    # ── Manual path entry ──────────────────────────────────────────
    "⌨  Enter path manually …": "__MANUAL__",
}


REPORT_DIR = DBDIR/"Blast"
REPORT_DIR.mkdir(exist_ok=True)

: 

In [ ]:
# Install dependencies if needed
# !pip install pandas ipywidgets matplotlib

# Initialize

In [ ]:
# ── CSS ───────────────────────────────────────────────────────────────────────
display(HTML("""
<style>
  .blast-header {
      background: linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);
      color:white; padding:18px 24px; border-radius:10px;
      font-family:'Segoe UI',sans-serif; margin-bottom:16px;
  }
  .blast-header h2 { margin:0 0 4px 0; font-size:1.5em; }
  .blast-header p  { margin:0; font-size:0.85em; opacity:0.75; }
  .section-title {
      font-weight:bold; font-size:0.9em; color:#4a5568;
      text-transform:uppercase; letter-spacing:0.05em;
      margin:14px 0 6px 0; font-family:'Segoe UI',sans-serif;
  }
  .hit-badge  {display:inline-block;background:#48bb78;color:white;padding:3px 10px;border-radius:12px;font-size:0.8em;font-weight:bold;margin-right:8px;}
  .warn-badge {display:inline-block;background:#ed8936;color:white;padding:3px 10px;border-radius:12px;font-size:0.8em;font-weight:bold;}
  .err-badge  {display:inline-block;background:#fc8181;color:white;padding:3px 10px;border-radius:12px;font-size:0.8em;font-weight:bold;}
  .info-badge {display:inline-block;background:#63b3ed;color:white;padding:3px 10px;border-radius:12px;font-size:0.8em;font-weight:bold;margin-right:6px;}
  .db-ok   {font-size:0.78em;color:#38a169;}
  .db-fasta{font-size:0.78em;color:#805ad5;}
  .db-miss {font-size:0.78em;color:#e53e3e;}
  .db-man  {font-size:0.78em;color:#718096;}
  .cmd-box {
      background:#1a202c;color:#9ae6b4;font-family:monospace;
      font-size:0.82em;padding:10px 14px;border-radius:8px;
      white-space:pre-wrap;margin:8px 0;
  }
  .stat-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin:12px 0;}
  .stat-card{background:#ebf8ff;border:1px solid #bee3f8;border-radius:8px;padding:10px 14px;text-align:center;}
  .stat-val {font-size:1.4em;font-weight:bold;color:#2b6cb0;}
  .stat-lbl {font-size:0.75em;color:#718096;margin-top:2px;}
  table.dataframe thead tr th{background:#2d3748!important;color:white!important;padding:8px 12px!important;font-size:0.82em;}
  table.dataframe tbody tr:nth-child(even){background:#f7fafc;}
  table.dataframe tbody tr:hover{background:#ebf8ff;}
  table.dataframe td{padding:6px 12px!important;font-size:0.82em;}
</style>
"""))

display(HTML("""
<div class='blast-header'>
  <h2>🧬 BLAST Runner</h2>
  <p>Supports pre-built BLAST databases and raw FASTA files (auto-indexed with makeblastdb)</p>
</div>
"""))



# ── DB helpers ────────────────────────────────────────────────────────────────
FASTA_EXTS = {".fasta", ".fa", ".fna", ".faa", ".ffn", ".frn", ".fsa"}

def path_kind(path):
    path = str(path) if path else ""
    if not path or path == "__MANUAL__":
        return "manual"
    ext = os.path.splitext(path)[1].lower()
    if ext in FASTA_EXTS:
        return "fasta" if os.path.isfile(path) else "missing"
    siblings = [path+".pin", path+".nin", path+".00.pin", path+".00.nin", path+".psq", path+".nsq"]
    if any(os.path.exists(s) for s in siblings):
        return "prebuilt"
    if os.path.isfile(path):
        return "fasta"
    return "missing"

def db_status_html(path):
    kind = path_kind(path)
    if kind == "prebuilt": return "<span class='db-ok'>✔ Pre-built BLAST database found</span>"
    if kind == "fasta":    return "<span class='db-fasta'>🔧 FASTA file — will run makeblastdb automatically</span>"
    if kind == "manual":   return "<span class='db-man'>✏ Enter a path below</span>"
    return "<span class='db-miss'>✗ Path not found — check the DATABASES dict above</span>"

def ensure_blastdb(fasta_path, dbtype):
    fasta_path = str(fasta_path)
    db_prefix = fasta_path
    if os.path.exists(fasta_path + ".pin") or os.path.exists(fasta_path + ".nin"):
        return db_prefix, False
    cmd = ["makeblastdb", "-in", fasta_path, "-dbtype", dbtype, "-out", db_prefix, "-parse_seqids"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"makeblastdb failed:\n{result.stderr}")
    return db_prefix, True

# ── Report helpers ────────────────────────────────────────────────────────────
def make_plots_b64(df):
    """Render the three plots and return a base64 PNG string."""
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    fig.patch.set_facecolor("#f8fafc")

    ax = axes[0]
    ax.hist(df["pident"], bins=20, color="#4299e1", edgecolor="white", linewidth=0.5)
    ax.set_title("% Identity Distribution", fontsize=10, fontweight="bold")
    ax.set_xlabel("% Identity"); ax.set_ylabel("Count")
    ax.set_facecolor("#f0f4f8")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[1]
    evals = df["evalue"].replace(0, 1e-300)
    log_e = -np.log10(evals.clip(1e-300))
    sc = ax.scatter(range(len(df)), log_e, c=df["pident"], cmap="RdYlGn", vmin=0, vmax=100, s=18, alpha=0.75)
    plt.colorbar(sc, ax=ax, label="% identity", pad=0.02)
    ax.set_title("-log₁₀(E-value) vs Hit Rank", fontsize=10, fontweight="bold")
    ax.set_xlabel("Hit rank"); ax.set_ylabel("-log₁₀(E-value)")
    ax.set_facecolor("#f0f4f8")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[2]
    top20 = df.head(20).copy()
    top20["label"] = top20["sseqid"].str[:22]
    colors = plt.cm.RdYlGn(top20["pident"].values / 100)
    ax.barh(range(len(top20)), top20["length"], color=colors, edgecolor="none")
    ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20["label"], fontsize=7)
    ax.invert_yaxis()
    ax.set_title("Alignment Length (top 20)", fontsize=10, fontweight="bold")
    ax.set_xlabel("Alignment length")
    ax.set_facecolor("#f0f4f8")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(0, 100))
    sm.set_array([]); plt.colorbar(sm, ax=ax, label="% identity", pad=0.02)

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(); buf.seek(0)
    return base64.b64encode(buf.read()).decode()

def make_plots(df):
    b64 = make_plots_b64(df)
    return f"<img src='data:image/png;base64,{b64}' style='max-width:100%;border-radius:8px;margin:8px 0'>"

def stats_html(df):
    best_e  = f"{df['evalue'].min():.2e}"
    avg_pid = f"{df['pident'].mean():.1f}%"
    avg_len = f"{df['length'].mean():.0f}"
    top = df.iloc[0]['sseqid']; top = top[:28]+"…" if len(top)>28 else top
    return (
        f"<div class='stat-grid'>"
        f"<div class='stat-card'><div class='stat-val'>{len(df)}</div><div class='stat-lbl'>Total hits</div></div>"
        f"<div class='stat-card'><div class='stat-val'>{best_e}</div><div class='stat-lbl'>Best E-value</div></div>"
        f"<div class='stat-card'><div class='stat-val'>{avg_pid}</div><div class='stat-lbl'>Avg identity</div></div>"
        f"<div class='stat-card'><div class='stat-val'>{avg_len}</div><div class='stat-lbl'>Avg aln length</div></div>"
        f"</div>"
        f"<div style='font-size:0.78em;color:#718096;margin-bottom:8px'>Top hit: <b>{top}</b></div>"
    )

def save_report(df, cmd, db_path, stamp):
    """Write a self-contained HTML report and a TSV to REPORT_DIR."""
    os.makedirs(REPORT_DIR, exist_ok=True)
    base = os.path.join(REPORT_DIR, stamp)

    # ── TSV ──────────────────────────────────────────────────────────────────
    tsv_path = base + ".tsv"
    df.to_csv(tsv_path, sep="\t", index=False)

    # ── HTML ─────────────────────────────────────────────────────────────────
    b64_plot = make_plots_b64(df)
    table_html = (
        df.style
        .background_gradient(subset=["pident"],   cmap="RdYlGn", vmin=0, vmax=100)
        .background_gradient(subset=["bitscore"], cmap="Blues")
        .format({"pident":"{:.1f}", "evalue":"{:.2e}", "bitscore":"{:.1f}"})
        .set_table_attributes('style="border-collapse:collapse;width:100%;font-size:0.82em;"')
        .to_html()
    )

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>BLAST Report — {stamp}</title>
<style>
  body {{ font-family:'Segoe UI',sans-serif; max-width:1200px; margin:40px auto; padding:0 24px; color:#2d3748; }}
  h1   {{ font-size:1.6em; margin-bottom:4px; }}
  .meta{{ font-size:0.82em; color:#718096; margin-bottom:24px; }}
  h2   {{ font-size:1em; font-weight:bold; color:#4a5568; text-transform:uppercase;
          letter-spacing:0.05em; margin:28px 0 10px; border-bottom:1px solid #e2e8f0; padding-bottom:4px; }}
  .stat-grid{{ display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin-bottom:20px; }}
  .stat-card{{ background:#ebf8ff; border:1px solid #bee3f8; border-radius:8px; padding:12px; text-align:center; }}
  .stat-val {{ font-size:1.5em; font-weight:bold; color:#2b6cb0; }}
  .stat-lbl {{ font-size:0.75em; color:#718096; margin-top:2px; }}
  .cmd-box  {{ background:#1a202c; color:#9ae6b4; font-family:monospace; font-size:0.82em;
               padding:12px 16px; border-radius:8px; white-space:pre-wrap; margin-bottom:20px; }}
  img       {{ max-width:100%; border-radius:8px; margin-bottom:20px; }}
  table     {{ border-collapse:collapse; width:100%; font-size:0.82em; }}
  thead th  {{ background:#2d3748; color:white; padding:8px 12px; text-align:left; }}
  tbody tr:nth-child(even) {{ background:#f7fafc; }}
  tbody tr:hover {{ background:#ebf8ff; }}
  td        {{ padding:6px 12px; }}
</style>
</head>
<body>
<h1>🧬 BLAST Report</h1>
<div class="meta">
  Generated: {stamp.replace("_", " ")} &nbsp;|&nbsp;
  Database: <strong>{os.path.basename(db_path)}</strong> &nbsp;|&nbsp;
  Program: <strong>{blast_type.value}</strong> &nbsp;|&nbsp;
  E-value: <strong>{evalue_slider.value}</strong>
</div>

<h2>Command</h2>
<div class="cmd-box">{" ".join(cmd)}</div>

<h2>Summary</h2>
{stats_html(df)}

<h2>Visualizations</h2>
<img src="data:image/png;base64,{b64_plot}">

<h2>Results Table ({len(df)} hits)</h2>
{table_html}
</body>
</html>"""

    html_path = base + ".html"
    with open(html_path, "w") as f:
        f.write(html)

    return tsv_path, html_path

# ── Example sequences ─────────────────────────────────────────────────────────
EXAMPLES = {
    "Protein (blastp)": (
        ">sp|P69905|HBA_HUMAN Hemoglobin subunit alpha\n"
        "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG\n"
        "KKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTP\n"
        "AVHASLDKFLASVSTVLTSKYR"
    ),
    "Nucleotide (blastn)": (
        ">NM_001101 Homo sapiens ACTB\n"
        "ATGGATGATGATATCGCCGCGCTCGTCGTCGACAACGGCTCCGGCATGTGCAAAGCCGGC\n"
        "TTCGCGGGCGACGATGCCCCGAGGCCGTCTTCCCCTCCATCGTGGGGCGCCCCAGGCACC"
    ),
    "Custom": ">query\nPASTE_YOUR_SEQUENCE_HERE",
}
PROGRAM_INFO = {
    "blastp":  "Protein query → Protein db",
    "blastn":  "Nucleotide query → Nucleotide db",
    "blastx":  "Nucleotide query (translated) → Protein db",
    "tblastn": "Protein query → Nucleotide db (translated)",
    "tblastx": "Nucleotide (translated) → Nucleotide db (translated)",
}
DBTYPE_FOR_PROGRAM = {
    "blastp": "prot", "blastx": "prot",
    "blastn": "nucl", "tblastn": "nucl", "tblastx": "nucl",
}

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — Input sequence
# ══════════════════════════════════════════════════════════════════════════════
display(HTML("<div class='section-title'>📋 Input Sequence</div>"))

example_picker = widgets.ToggleButtons(
    options=list(EXAMPLES.keys()), description="",
    tooltips=["Human hemoglobin alpha", "Human ACTB mRNA", "Paste your own"],
    layout=widgets.Layout(margin="0 0 8px 0"),
)
sequence_box = widgets.Textarea(
    value=EXAMPLES["Protein (blastp)"],
    placeholder="Paste FASTA sequence here…",
    layout=widgets.Layout(width="100%", height="130px"),
)
seq_counter = widgets.HTML()

def update_counter(change=None):
    txt = sequence_box.value
    seq = "".join(l for l in txt.splitlines() if not l.startswith(">")).replace(" ", "")
    hdrs = sum(1 for l in txt.splitlines() if l.startswith(">"))
    seq_counter.value = (
        f"<span style='font-size:0.78em;color:#718096;'>"
        f"{hdrs} sequence(s) · {len(seq)} residues/bases</span>"
    )

def on_example_change(change):
    sequence_box.value = EXAMPLES[change["new"]]
    if change["new"] == "Nucleotide (blastn)": blast_type.value = "blastn"
    elif change["new"] == "Protein (blastp)":  blast_type.value = "blastp"

example_picker.observe(on_example_change, names="value")
sequence_box.observe(update_counter, names="value")
update_counter()
display(example_picker, sequence_box, seq_counter)

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — Database selection
# ══════════════════════════════════════════════════════════════════════════════
display(HTML("<div class='section-title'>🗄 Database</div>"))

db_dropdown = widgets.Dropdown(
    options=list(DATABASES.keys()),
    description="Database:",
    layout=widgets.Layout(width="480px"),
)
db_status = widgets.HTML()
manual_path = widgets.Text(
    placeholder="/path/to/db_prefix  or  /path/to/seqs.fasta",
    description="Path:",
    layout=widgets.Layout(width="480px"),
)
manual_row = widgets.HBox([manual_path])

def get_selected_path():
    raw = DATABASES.get(db_dropdown.value, "")
    if raw == "__MANUAL__":
        return str(manual_path.value.strip())
    return str(raw)

def refresh_db_status(change=None):
    path = get_selected_path()
    manual_row.layout.display = "flex" if DATABASES.get(db_dropdown.value) == "__MANUAL__" else "none"
    db_status.value = db_status_html(path)

db_dropdown.observe(refresh_db_status, names="value")
manual_path.observe(refresh_db_status, names="value")
refresh_db_status()
display(db_dropdown, manual_row, db_status)

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Search settings
# ══════════════════════════════════════════════════════════════════════════════
display(HTML("<div class='section-title'>⚙️ Search Settings</div>"))

blast_type = widgets.Dropdown(
    options=list(PROGRAM_INFO.keys()), value="blastp",
    description="Program:", layout=widgets.Layout(width="220px"),
)
program_hint = widgets.HTML()

def update_hint(change=None):
    program_hint.value = (
        f"<span style='font-size:0.78em;color:#718096;'>{PROGRAM_INFO[blast_type.value]}</span>"
    )

blast_type.observe(update_hint, names="value")
update_hint()

evalue_slider = widgets.SelectionSlider(
    options=[1e-100, 1e-50, 1e-20, 1e-10, 1e-5, 1e-3, 0.01, 0.1, 1, 10],
    value=1e-5, description="E-value:", readout=True,
    layout=widgets.Layout(width="420px"),
)
max_hits = widgets.BoundedIntText(
    value=50, min=1, max=5000,
    description="Max hits:", layout=widgets.Layout(width="200px"),
)
threads_box = widgets.BoundedIntText(
    value=4, min=1, max=64,
    description="Threads:", layout=widgets.Layout(width="180px"),
)
pident_filter = widgets.BoundedFloatText(
    value=0.0, min=0.0, max=100.0, step=1.0,
    description="Min %id:", layout=widgets.Layout(width="180px"),
)

display(
    widgets.HBox([blast_type, program_hint], layout=widgets.Layout(align_items="center", gap="12px")),
    evalue_slider,
    widgets.HBox([max_hits, threads_box, pident_filter], layout=widgets.Layout(gap="16px")),
)

# ── Action buttons ────────────────────────────────────────────────────────────
display(HTML("<div style='margin-top:10px'></div>"))
run_button    = widgets.Button(description=" ▶  Run BLAST",  button_style="success", layout=widgets.Layout(width="155px", height="36px"))
clear_button  = widgets.Button(description=" ✕  Clear",      button_style="warning", layout=widgets.Layout(width="105px", height="36px"))
export_button = widgets.Button(description=" ⬇  Export TSV", button_style="info",    disabled=True, layout=widgets.Layout(width="140px", height="36px"))
status_label  = widgets.HTML()
display(widgets.HBox([run_button, clear_button, export_button, status_label],
                     layout=widgets.Layout(align_items="center", gap="10px")))

output = widgets.Output()
display(output)

_last_df = None

# ── Run BLAST ─────────────────────────────────────────────────────────────────
def run_blast(btn):
    global _last_df
    with output:
        clear_output(wait=True)
        run_button.disabled = True
        export_button.disabled = True
        status_label.value = "<span style='color:#3182ce;font-size:0.85em'>⏳ Running…</span>"

        fasta_text = sequence_box.value.strip()
        if not fasta_text.startswith(">"):
            display(HTML("<span class='err-badge'>✗ Error</span> Input must be in FASTA format."))
            run_button.disabled = False
            status_label.value = ""
            return

        # Show query sequence
        display(HTML(
            f"""
            <div class='section-title'>🔍 Query Sequence</div>
            <pre style='background:#f7fafc;
                        padding:10px;
                        border-radius:8px;
                        max-height:250px;
                        overflow:auto;
                        font-size:0.8em;'>
        {fasta_text}
            </pre>
            """
        ))

        db_path = get_selected_path()
        if not db_path:
            display(HTML("<span class='err-badge'>✗ No database</span> Select or enter a database path."))
            run_button.disabled = False; status_label.value = ""
            return

        fasta_file = out_file = None
        try:
            kind = path_kind(db_path)
            if kind == "missing":
                display(HTML(f"<span class='err-badge'>✗ Path not found</span> <code>{db_path}</code>"))
                return

            actual_db = db_path
            if kind == "fasta":
                dbtype = DBTYPE_FOR_PROGRAM[blast_type.value]
                status_label.value = "<span style='color:#805ad5;font-size:0.85em'>🔧 Indexing FASTA…</span>"
                actual_db, built = ensure_blastdb(db_path, dbtype)
                msg = "makeblastdb ran" if built else "Reusing existing index"
                display(HTML(
                    f"<span class='info-badge'>ℹ {'Indexed' if built else 'Reusing index'}</span> "
                    f"{msg} for <code>{os.path.basename(db_path)}</code>"
                ))
                status_label.value = "<span style='color:#3182ce;font-size:0.85em'>⏳ Running BLAST…</span>"

            with tempfile.NamedTemporaryFile(mode="w", suffix=".fasta", delete=False) as f:
                f.write(fasta_text); fasta_file = f.name
            out_file = fasta_file + ".tsv"

            cmd = [
                blast_type.value,
                "-query",           fasta_file,
                "-db",              actual_db,
                "-evalue",          str(evalue_slider.value),
                "-max_target_seqs", str(max_hits.value),
                "-num_threads",     str(threads_box.value),
                "-outfmt",
                "6 qseqid sseqid pident length mismatch gapopen "
                "qstart qend sstart send evalue bitscore",
                "-out", out_file,
            ]

            display(HTML(
                f"<div class='section-title'>🖥 Command</div>"
                f"<div class='cmd-box'>{' '.join(cmd)}</div>"
            ))

            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode != 0:
                display(HTML(
                    f"<span class='err-badge'>✗ BLAST failed</span>"
                    f"<pre style='color:#c53030;font-size:0.8em;margin-top:6px'>{result.stderr}</pre>"
                ))
                return

            cols = ["qseqid","sseqid","pident","length","mismatch","gapopen",
                    "qstart","qend","sstart","send","evalue","bitscore"]
            df = pd.read_csv(out_file, sep="\t", names=cols)

            if df.empty:
                display(HTML("<span class='warn-badge'>⚠ No hits</span> Try relaxing the E-value threshold."))
                return

            if pident_filter.value > 0:
                df = df[df["pident"] >= pident_filter.value]
                if df.empty:
                    display(HTML(f"<span class='warn-badge'>⚠ No hits</span> after ≥{pident_filter.value:.0f}% identity filter."))
                    return

            _last_df = df
            export_button.disabled = False

            # ── Save report ──────────────────────────────────────────────────
            stamp = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_{blast_type.value}_{os.path.basename(db_path)}"
            stamp = stamp.replace(" ", "_")
            tsv_path, html_path = save_report(df, cmd, db_path, stamp)

            # ── Display results ───────────────────────────────────────────────
            display(HTML(
                f"<div class='section-title'>📊 Summary</div>"
                f"<span class='hit-badge'>✓ {len(df)} hits</span>" + stats_html(df)
            ))
            display(HTML("<div class='section-title'>📈 Visualizations</div>" + make_plots(df)))

            styled = (
                df.style
                .background_gradient(subset=["pident"],   cmap="RdYlGn", vmin=0, vmax=100)
                .background_gradient(subset=["bitscore"], cmap="Blues")
                .format({"pident":"{:.1f}", "evalue":"{:.2e}", "bitscore":"{:.1f}"})
                .set_table_styles([{"selector":"thead tr th",
                    "props":[("background-color","#2d3748"),("color","white"),
                             ("font-size","0.82em"),("padding","8px 12px")]}])
            )
            display(HTML("<div class='section-title'>🗃 Results Table</div>"))
            display(styled)

            display(HTML(
                f"<div style='margin-top:14px;font-size:0.82em;color:#4a5568;'>"
                f"💾 Saved to <code>{os.path.abspath(REPORT_DIR)}/</code> &nbsp;·&nbsp; "
                f"<code>{os.path.basename(tsv_path)}</code> &nbsp;+&nbsp; "
                f"<code>{os.path.basename(html_path)}</code></div>"
            ))

            status_label.value = f"<span style='color:#38a169;font-size:0.85em'>✓ Done — {len(df)} hits</span>"

        except FileNotFoundError as e:
            prog = str(e).split("'")[1] if "'" in str(e) else blast_type.value
            display(HTML(f"<span class='err-badge'>✗ Not found</span> <code>{prog}</code> is not on PATH. Is BLAST+ installed?"))
            status_label.value = ""
        except RuntimeError as e:
            display(HTML(f"<span class='err-badge'>✗ Error</span><pre style='font-size:0.8em'>{e}</pre>"))
            status_label.value = ""
        except Exception as e:
            display(HTML(f"<span class='err-badge'>✗ Error</span> <code>{e}</code>"))
            status_label.value = ""
        finally:
            for p in [fasta_file, out_file]:
                if p and os.path.exists(p): os.remove(p)
            run_button.disabled = False

# ── Export / Clear ────────────────────────────────────────────────────────────
def export_tsv(btn):
    """Manual export button — saves just the TSV if user wants an extra copy."""
    if _last_df is None: return
    os.makedirs(REPORT_DIR, exist_ok=True)
    out = os.path.join(REPORT_DIR, "blast_results_latest.tsv")
    _last_df.to_csv(out, sep="\t", index=False)
    with output:
        display(HTML(f"<span class='hit-badge'>⬇ Saved</span> <code>{os.path.abspath(out)}</code>"))

def clear_output_panel(btn):
    global _last_df
    with output: clear_output()
    _last_df = None
    export_button.disabled = True
    status_label.value = ""

run_button.on_click(run_blast)
export_button.on_click(export_tsv)
clear_button.on_click(clear_output_panel)